# Prediksi Debit Aliran Sungai dan Risiko Banjir Berbasis Curah Hujan
### Gradient Boosting Regressor dengan Data Multi-Sumber (Open-Meteo Geocoding, Open-Meteo Flood, NASA POWER)

**Apa yang terjadi:** notebook ini membangun model machine learning untuk memprediksi debit sungai 1 hari ke depan di kota-kota rawan banjir Indonesia, lalu menyiapkan semua aset yang dibutuhkan dashboard Streamlit.

**Keterangan:**
- Data berasal dari 3 tabel hasil pipeline ETL Airflow: `locations.csv`, `river_discharge.csv`, `rainfall_daily.csv`.
- Debit (GloFAS) dan hujan (NASA POWER) adalah estimasi model dan satelit, bukan pengukuran stasiun lapangan. Hasil notebook ini untuk pembelajaran, bukan sistem peringatan resmi.

## Lima Pertanyaan Bisnis

Lima pertanyaan berikut memandu seluruh analisis. Setiap bagian EDA dan pemodelan menjawab satu pertanyaan, dan jawabannya dicetak otomatis dari data.

| No | Tipe | Fokus | Pertanyaan Bisnis |
|:---|:---|:---|:---|
| **1** | **Dasar** | Gambaran umum data | Bagaimana gambaran umum data dan karakter debit sungai serta curah hujan di tiap kota rawan banjir? |
| **2** | **Analitik (masalah)** | Tingkat risiko | Kota mana yang paling berisiko banjir, dilihat dari seberapa ekstrem puncak debit dan seberapa sering debit melonjak? |
| **3** | **Analitik (urgensi)** | Waktu peringatan | Seberapa cepat debit naik setelah hujan lebat, sehingga berapa hari waktu peringatan dini yang tersedia? |
| **4** | **Analitik (pola dan pemicu)** | Musiman dan korelasi | Bagaimana pola musiman debit, bulan apa yang paling berisiko, dan seberapa kuat hubungan hujan dengan debit? |
| **5** | **AI (metode, algoritma)** | Akurasi dan keandalan model | Algoritma apa yang paling akurat memprediksi debit 1 hari ke depan, fitur apa yang paling berpengaruh, dan apakah model andal mendeteksi kondisi di atas ambang siaga? |

## Tahap 1: Import Library

**Apa yang terjadi:** seluruh library yang dibutuhkan sampai tahap akhir diimpor dalam satu sel, termasuk pengaturan tampilan grafik.

**Keterangan:**
- `RANDOM_STATE` dipakai di seluruh proses agar hasil bisa diulang.
- Import khusus Colab (`drive`, `files`) dibungkus `try/except` sehingga notebook tetap jalan di lingkungan lain.

In [ ]:
import os
import json
import shutil
import warnings
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import sklearn
from IPython.display import display

from sklearn.base import clone
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.inspection import permutation_importance
from sklearn.metrics import (ConfusionMatrixDisplay, classification_report, confusion_matrix,
                             f1_score, mean_absolute_error, mean_squared_error,
                             precision_score, r2_score, recall_score)
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit

try:
    from google.colab import drive, files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["font.sans-serif"] = "DejaVu Sans"
plt.rcParams["figure.figsize"] = (11, 5)
plt.rcParams["figure.dpi"] = 110
RANDOM_STATE = 42

print("Seluruh library berhasil diimpor. scikit-learn", sklearn.__version__, "| Colab:", IN_COLAB)

## Tahap 2: Penilaian Data (Assessing) dan Pembersihan 3 Tabel

### 2.1 Memuat data dari Google Drive
**Apa yang terjadi:** Google Drive dihubungkan, lalu 3 file CSV dicari di beberapa folder umum dan dibaca.

**Keterangan:** simpan ketiga CSV dalam satu folder, misalnya `MyDrive/banjir`. Jika foldermu berbeda, tambahkan ke `search_dirs`. Jika file tidak ditemukan, notebook berhenti dengan pesan yang jelas.

In [ ]:
if IN_COLAB:
    drive.mount("/content/drive")

search_dirs = [
    "/content/drive/MyDrive/banjir",
    "/content/drive/MyDrive/flood_data",
    "/content/drive/MyDrive/debit_sungai",
    "/content/drive/MyDrive",
    ".",
    "../",
]

def locate_csv(filename):
    for d in search_dirs:
        path = os.path.join(d, filename)
        if os.path.exists(path):
            return path
    raise FileNotFoundError(f"{filename} tidak ditemukan di: {search_dirs}")

loc_path = locate_csv("locations.csv")
dis_path = locate_csv("river_discharge.csv")
rain_path = locate_csv("rainfall_daily.csv")

df_loc_raw = pd.read_csv(loc_path)
df_dis_raw = pd.read_csv(dis_path)
df_rain_raw = pd.read_csv(rain_path)

BASE_DIR = os.path.dirname(os.path.abspath(loc_path))   # folder keluaran model dan aset dashboard
print("Folder data:", BASE_DIR)
print(f"locations: {len(df_loc_raw)} baris | river_discharge: {len(df_dis_raw)} baris | rainfall_daily: {len(df_rain_raw)} baris")

### 2.2 Assessing: head, info, describe, nilai kosong, dan duplikat
**Apa yang terjadi:** setiap tabel diperiksa dengan `head()`, `info()`, `describe()`, jumlah nilai kosong, duplikat baris penuh, dan duplikat kunci.

**Keterangan:** *assessing* dilakukan **sebelum** data diubah agar masalahnya terdokumentasi. Kunci unik: `location_id` pada `locations`, serta `location_id` + `obs_date` pada dua tabel lainnya.

In [ ]:
tables = {
    "locations": (df_loc_raw, ["location_id"]),
    "river_discharge": (df_dis_raw, ["location_id", "obs_date"]),
    "rainfall_daily": (df_rain_raw, ["location_id", "obs_date"]),
}

for name, (d, key) in tables.items():
    print("=" * 80)
    print(f"{name.upper()}  |  {d.shape[0]} baris x {d.shape[1]} kolom")
    print("=" * 80)
    display(d.head())
    print("\n-- df.info() --")
    d.info()
    print("\n-- df.describe() --")
    display(d.describe(include="all").T)
    kosong = d.isna().sum()
    print("\n-- Nilai kosong per kolom --")
    print(kosong[kosong > 0].to_string() if kosong.sum() else "Tidak ada nilai kosong")
    print(f"\nDuplikat baris penuh : {d.duplicated().sum()}")
    print(f"Duplikat kunci {key} : {d.duplicated(subset=key).sum()}\n")

In [ ]:
# Konsistensi relasi dan kualitas nilai
print("Orphan river_discharge (kota tidak ada di locations):", (~df_dis_raw["location_id"].isin(df_loc_raw["location_id"])).sum())
print("Orphan rainfall_daily  (kota tidak ada di locations):", (~df_rain_raw["location_id"].isin(df_loc_raw["location_id"])).sum())
print("Lokasi tanpa data debit:", sorted(set(df_loc_raw["location_id"]) - set(df_dis_raw["location_id"])))
print("Debit <= 0                 :", (df_dis_raw["discharge_m3s"] <= 0).sum())
print("Hujan negatif (sentinel -999):", (df_rain_raw["rain_mm"] < 0).sum())

def coverage(df, name):
    d = df.assign(obs_date=pd.to_datetime(df["obs_date"]))
    c = d.groupby("location_id")["obs_date"].agg(awal="min", akhir="max", jumlah="count")
    c["hari_hilang"] = (c["akhir"] - c["awal"]).dt.days + 1 - c["jumlah"]
    print(f"\nRentang dan celah tanggal: {name} (ringkasan)")
    display(c.head(8))
    print("Total hari hilang:", int(c["hari_hilang"].sum()))

coverage(df_dis_raw, "river_discharge")
coverage(df_rain_raw, "rainfall_daily")

### 2.3 Data wrangling: cleaning per tabel
**Apa yang terjadi:** tipe data diperbaiki, teks dirapikan, nilai sentinel `-999` pada hujan diubah menjadi kosong lalu ditambal per kota, baris tidak valid dibuang, dan duplikat kunci dihapus. Setelah itu data diperiksa ulang.

**Keterangan:**
- Cleaning dibagi dua tahap: **sebelum join** (masalah milik tabel sendiri, termasuk duplikat kunci yang akan menggandakan baris saat join) dan **sesudah join** (celah tanggal, di Tahap 3).
- **Outlier debit tidak dibuang** karena nilai ekstrem adalah kejadian banjir yang ingin diprediksi.

In [ ]:
df_loc = df_loc_raw.copy()
df_dis = df_dis_raw.copy()
df_rain = df_rain_raw.copy()

for d in (df_loc, df_dis, df_rain):
    d["location_id"] = d["location_id"].astype(str).str.strip()
for c in ["city_name", "river_name", "province"]:
    if c in df_loc.columns:
        df_loc[c] = df_loc[c].astype(str).str.strip()

df_dis["obs_date"] = pd.to_datetime(df_dis["obs_date"], errors="coerce")
df_rain["obs_date"] = pd.to_datetime(df_rain["obs_date"], errors="coerce")
df_dis["discharge_m3s"] = pd.to_numeric(df_dis["discharge_m3s"], errors="coerce")
df_rain["rain_mm"] = pd.to_numeric(df_rain["rain_mm"], errors="coerce")

# Sentinel -999 -> NaN -> tambal celah pendek per kota
df_rain["rain_mm"] = df_rain["rain_mm"].where(df_rain["rain_mm"] > -900)
df_rain = df_rain.sort_values(["location_id", "obs_date"])
df_rain["rain_mm"] = df_rain.groupby("location_id")["rain_mm"].transform(lambda g: g.interpolate(limit=3))
df_rain.loc[df_rain["rain_mm"] < 0, "rain_mm"] = np.nan

df_dis = df_dis.dropna(subset=["obs_date", "discharge_m3s"])
df_dis = df_dis[df_dis["discharge_m3s"] >= 0]
df_rain = df_rain.dropna(subset=["obs_date", "rain_mm"])

df_loc = df_loc.drop_duplicates(subset="location_id", keep="last")
df_dis = df_dis.sort_values(["location_id", "obs_date"]).drop_duplicates(subset=["location_id", "obs_date"], keep="last")
df_rain = df_rain.sort_values(["location_id", "obs_date"]).drop_duplicates(subset=["location_id", "obs_date"], keep="last")

# Re-check
print(f"locations       : {len(df_loc_raw)} -> {len(df_loc)} baris | duplikat kunci: {df_loc.duplicated(subset=['location_id']).sum()}")
print(f"river_discharge : {len(df_dis_raw)} -> {len(df_dis)} baris | duplikat kunci: {df_dis.duplicated(subset=['location_id','obs_date']).sum()} | kosong: {int(df_dis.isna().sum().sum())}")
print(f"rainfall_daily  : {len(df_rain_raw)} -> {len(df_rain)} baris | duplikat kunci: {df_rain.duplicated(subset=['location_id','obs_date']).sum()} | kosong: {int(df_rain.isna().sum().sum())}")

## Tahap 3: Penggabungan 3 Tabel dan Penilaian Ulang

### 3.1 Join
**Apa yang terjadi:** `river_discharge` digabung dengan `rainfall_daily` lewat `location_id` + `obs_date` (satu ke satu), lalu dengan `locations` lewat `location_id` (banyak ke satu).

**Keterangan:** parameter `validate` memastikan relasi benar sehingga baris tidak berlipat diam-diam. `inner join` dipakai untuk debit dan hujan karena baris tanpa pasangan hujan tidak bisa dipakai model.

In [ ]:
df_merged = pd.merge(df_dis, df_rain, on=["location_id", "obs_date"], how="inner", validate="1:1")
df_merged = pd.merge(df_merged, df_loc, on="location_id", how="left", validate="m:1")

print("Shape hasil gabungan:", df_merged.shape)
print("Baris debit tanpa pasangan hujan (terbuang oleh inner join):", len(df_dis) - len(df_merged))
display(df_merged.head())

### 3.2 Assessing data gabungan
**Apa yang terjadi:** data gabungan diperiksa ulang: `info()`, `describe()`, nilai kosong, duplikat baris dan kunci, serta celah tanggal per kota.

**Keterangan:** celah tanggal penting karena fitur lag memakai `shift`, yang hanya benar jika satu baris adalah satu hari berurutan.

In [ ]:
df_merged.info()
print()
display(df_merged[["discharge_m3s", "rain_mm"]].describe().T)

kosong = df_merged.isna().sum()
print("\nNilai kosong per kolom:")
print(kosong[kosong > 0].to_string() if kosong.sum() else "Tidak ada nilai kosong")
print("\nDuplikat baris penuh:", df_merged.duplicated().sum())
print("Duplikat kunci (location_id, obs_date):", df_merged.duplicated(subset=["location_id", "obs_date"]).sum())

gap = (df_merged.groupby("location_id")["obs_date"]
       .agg(lambda s: (s.max() - s.min()).days + 1 - s.nunique()))
print("\nKota dengan celah tanggal:", int((gap > 0).sum()), "dari", len(gap), "| total hari hilang:", int(gap.sum()))

### 3.3 Cleaning pasca-join
**Apa yang terjadi:** setiap kota dijadikan deret harian tanpa celah (celah pendek maksimal 3 hari ditambal interpolasi), sisa baris kosong dibuang, dan kota dengan data terlalu sedikit dikeluarkan. Hasilnya diperiksa ulang.

**Keterangan:** ambang minimal 120 hari per kota (`MIN_DAYS`) menjaga agar fitur lag dan rolling punya cukup data. Outlier hanya dilaporkan, tidak dibuang.

In [ ]:
STATIC_COLS = [c for c in df_loc.columns if c != "location_id"]
NUM_COLS = ["discharge_m3s", "rain_mm"]
MIN_DAYS = 120

parts = []
for loc_id, g in df_merged.groupby("location_id"):
    g = g.set_index("obs_date").sort_index()
    g = g.reindex(pd.date_range(g.index.min(), g.index.max(), freq="D"))
    g[NUM_COLS] = g[NUM_COLS].interpolate(limit=3, limit_area="inside")
    g[STATIC_COLS] = g[STATIC_COLS].ffill().bfill()
    g["location_id"] = loc_id
    g.index.name = "obs_date"
    parts.append(g.reset_index())

df = pd.concat(parts, ignore_index=True).dropna(subset=NUM_COLS)
df = df.sort_values(["location_id", "obs_date"]).reset_index(drop=True)

cnt = df.groupby("location_id").size()
kurang = cnt[cnt < MIN_DAYS].index.tolist()
if kurang:
    print(f"Kota dibuang karena < {MIN_DAYS} hari data:", kurang)
    df = df[~df["location_id"].isin(kurang)].reset_index(drop=True)

# Re-check
print(f"Baris: {len(df_merged)} -> {len(df)} | kota: {df['city_name'].nunique()}")
print("Nilai kosong (kolom numerik):", int(df[NUM_COLS].isna().sum().sum()))
print("Duplikat kunci:", df.duplicated(subset=["location_id", "obs_date"]).sum())

def iqr_outlier_pct(s):
    q1, q3 = s.quantile([0.25, 0.75]); iqr = q3 - q1
    return ((s < q1 - 1.5 * iqr) | (s > q3 + 1.5 * iqr)).mean() * 100
out = df.groupby("city_name")[NUM_COLS].agg(iqr_outlier_pct).round(1)
out.columns = ["debit_outlier_%", "hujan_outlier_%"]
print("\n10 kota dengan persentase outlier debit tertinggi (DIPERTAHANKAN):")
display(out.sort_values("debit_outlier_%", ascending=False).head(10))

## Tahap 4: Exploratory Data Analysis (4 Grafik untuk Pertanyaan 1 sampai 4)

Satu grafik menjawab satu pertanyaan bisnis. Jawaban dicetak otomatis di bawah tiap grafik.

### EDA 1 (Pertanyaan 1): karakter debit dan hujan tiap kota
**Apa yang terjadi:** tabel ringkasan per kota dibuat, lalu divisualkan dalam *bubble chart*.

**Keterangan:** sumbu X = hujan rata-rata, sumbu Y = debit rata-rata (skala log), ukuran gelembung = debit maksimum. Kota di kanan atas basah dan berdebit besar; gelembung besar berarti puncak debitnya tinggi.

In [ ]:
n_city = df["city_name"].nunique()

summary = df.groupby("city_name").agg(
    hari=("obs_date", "count"),
    debit_rata=("discharge_m3s", "mean"),
    debit_maks=("discharge_m3s", "max"),
    hujan_rata=("rain_mm", "mean"),
    hujan_maks=("rain_mm", "max"),
    hari_hujan_lebat=("rain_mm", lambda s: int((s >= 20).sum())),
).round(2)
display(summary.sort_values("debit_rata", ascending=False))

sm = summary.reset_index()
plt.figure(figsize=(12, 7))
sns.scatterplot(data=sm, x="hujan_rata", y="debit_rata", size="debit_maks",
                sizes=(60, 900), alpha=0.55, color="tab:blue", legend=False)
for _, r in sm.iterrows():
    plt.annotate(r["city_name"], (r["hujan_rata"], r["debit_rata"]), fontsize=8,
                 xytext=(5, 5), textcoords="offset points")
plt.yscale("log")
plt.title("EDA 1: Karakter debit dan hujan per kota (ukuran gelembung = debit maksimum)")
plt.xlabel("Curah hujan rata-rata (mm/hari)")
plt.ylabel("Debit rata-rata (m³/detik, skala log)")
plt.tight_layout(); plt.show()

print(f"JAWABAN 1: {n_city} kota, {len(df):,} baris harian, {df['obs_date'].min().date()} s.d. {df['obs_date'].max().date()}.")
print(f"- Debit rata-rata tertinggi: {summary['debit_rata'].idxmax()} ({summary['debit_rata'].max():.1f} m³/s)")
print(f"- Hujan rata-rata tertinggi: {summary['hujan_rata'].idxmax()} ({summary['hujan_rata'].max():.1f} mm/hari)")
print(f"- Hari hujan lebat (>= 20 mm) terbanyak: {summary['hari_hujan_lebat'].idxmax()} ({summary['hari_hujan_lebat'].max()} hari)")

### EDA 2 (Pertanyaan 2): kota paling berisiko banjir
**Apa yang terjadi:** debit dinormalisasi terhadap median kotanya sendiri, lalu empat indikator risiko dihitung dan dirangkum menjadi skor risiko.

**Keterangan:** indikator = rasio puncak (maksimum/median), CV (fluktuasi), hari lonjakan (naik >= 25% dari hari sebelumnya), dan hari di atas 2x median. Skor risiko adalah rata-rata peringkat keempatnya. Grafik: batang horizontal urut dari paling berisiko, dengan angka rasio puncak.

In [ ]:
df["rasio_median"] = df["discharge_m3s"] / df.groupby("city_name")["discharge_m3s"].transform("median")
df["pct_change"] = df.groupby("location_id")["discharge_m3s"].pct_change()

risk = df.groupby("city_name").agg(
    rasio_puncak=("rasio_median", "max"),
    cv=("discharge_m3s", lambda s: s.std() / s.mean()),
    hari_lonjakan=("pct_change", lambda s: int((s >= 0.25).sum())),
    hari_di_atas_2x_median=("rasio_median", lambda s: int((s >= 2).sum())),
)
risk["skor_risiko"] = risk.rank(ascending=True).mean(axis=1)
risk = risk.sort_values("skor_risiko", ascending=False).round(2)
display(risk.head(10))

r = risk["skor_risiko"].sort_values()
fig, ax = plt.subplots(figsize=(10, max(5, 0.32 * n_city + 2)))
ax.barh(r.index, r.values, color=plt.cm.Reds(np.linspace(0.3, 0.95, len(r))))
for y, (city, val) in enumerate(r.items()):
    ax.text(val + 0.1, y, f"puncak {risk.loc[city, 'rasio_puncak']:.1f}x median", va="center", fontsize=8)
ax.set_title("EDA 2: Skor risiko banjir per kota (makin tinggi makin berisiko)")
ax.set_xlabel("Skor risiko gabungan (rata-rata peringkat 4 indikator)")
plt.tight_layout(); plt.show()

print(f"JAWABAN 2: Kota paling berisiko: {risk.index[0]} (puncak debit {risk.iloc[0]['rasio_puncak']:.1f}x median).")
print("- 5 teratas:", " > ".join(risk.index[:5]))

### EDA 3 (Pertanyaan 3): seberapa cepat debit naik setelah hujan lebat
**Apa yang terjadi:** hari-hari dengan hujan sangat lebat (persentil ke-95 tiap kota) diambil sebagai "kejadian". Untuk tiap kejadian dihitung rasio debit pada hari 0 sampai +7 terhadap debit sehari sebelum hujan, lalu dirata-rata per kota.

**Keterangan:** garis = median seluruh kota, area biru = rentang antar-kuartil (25% sampai 75% kota). Nilai 1 berarti debit tidak berubah. Hari puncak kurva adalah perkiraan waktu peringatan dini yang tersedia.

In [ ]:
rows = []
for city, g in df.sort_values("obs_date").groupby("city_name"):
    g = g.reset_index(drop=True)
    thr_rain = g["rain_mm"].quantile(0.95)
    for i in g.index[g["rain_mm"] >= thr_rain]:
        if i - 1 >= 0 and i + 7 < len(g) and g.loc[i - 1, "discharge_m3s"] > 0:
            base = g.loc[i - 1, "discharge_m3s"]
            for k in range(-1, 8):
                rows.append((city, k, g.loc[i + k, "discharge_m3s"] / base))
ev = pd.DataFrame(rows, columns=["city_name", "hari_ke", "rasio_debit"])
piv = ev.groupby(["city_name", "hari_ke"])["rasio_debit"].mean().unstack()

plt.figure(figsize=(11, 5.5))
plt.fill_between(piv.columns, piv.quantile(0.25), piv.quantile(0.75), alpha=0.25, color="tab:blue", label="Rentang antar-kuartil kota")
plt.plot(piv.columns, piv.median(), marker="o", color="tab:blue", lw=2, label="Median seluruh kota")
plt.axhline(1, color="k", ls="--", lw=1); plt.axvline(0, color="gray", ls=":", lw=1)
plt.title("EDA 3: Respons debit setelah hari hujan sangat lebat (hari 0)")
plt.xlabel("Hari relatif terhadap hujan lebat"); plt.ylabel("Debit / debit sehari sebelum hujan")
plt.legend(); plt.tight_layout(); plt.show()

peak_day = piv.idxmax(axis=1)
print(f"JAWABAN 3: Debit rata-rata mencapai puncak {peak_day.mean():.1f} hari setelah hujan sangat lebat "
      f"(tercepat {peak_day.min()} hari, terlama {peak_day.max()} hari).")
print("- Sebaran hari puncak (jumlah kota):", peak_day.value_counts().sort_index().to_dict())
print(f"- Median debit pada puncak: {piv.median().max():.2f}x debit sebelum hujan.")

### EDA 4 (Pertanyaan 4): pola musiman dan hubungan hujan dengan debit
**Apa yang terjadi:** debit relatif (terhadap rata-rata kotanya) dan hujan dirata-rata per bulan di seluruh kota. Kekuatan hubungan hujan-debit diukur dengan korelasi Spearman.

**Keterangan:** garis merah = debit relatif (1,0 = rata-rata kota), batang biru = hujan rata-rata. Jika data hanya sekitar 1 tahun, tiap bulan muncul sekali sehingga pola musiman belum kuat. Menambah `HISTORY_DAYS` di DAG (mis. 1095) akan memperkuatnya.

In [ ]:
df["bulan"] = df["obs_date"].dt.month
df["debit_norm"] = df["discharge_m3s"] / df.groupby("city_name")["discharge_m3s"].transform("mean")
df["hujan_7h"] = df.groupby("location_id")["rain_mm"].transform(lambda s: s.rolling(7).sum())

bln = df.groupby("bulan").agg(debit_relatif=("debit_norm", "mean"), hujan=("rain_mm", "mean"))
r7 = df[["discharge_m3s", "hujan_7h"]].corr(method="spearman").iloc[0, 1]
r1 = df[["discharge_m3s", "rain_mm"]].corr(method="spearman").iloc[0, 1]

fig, ax = plt.subplots(figsize=(11, 5.5))
ax2 = ax.twinx()
ax2.bar(bln.index, bln["hujan"], color="tab:blue", alpha=0.3, label="Hujan rata-rata (mm/hari)")
ax2.set_ylabel("Hujan rata-rata (mm/hari)"); ax2.grid(False)
ax.set_zorder(ax2.get_zorder() + 1); ax.patch.set_visible(False)
ax.plot(bln.index, bln["debit_relatif"], marker="o", color="tab:red", lw=2, label="Debit relatif")
ax.axhline(1, color="k", ls="--", lw=1)
ax.set_xticks(range(1, 13)); ax.set_xlabel("Bulan"); ax.set_ylabel("Debit relatif (1,0 = rata-rata kota)")
ax.set_title(f"EDA 4: Pola musiman debit dan hujan | korelasi Spearman hujan 7 hari vs debit = {r7:.2f}")
h1, l1 = ax.get_legend_handles_labels(); h2, l2 = ax2.get_legend_handles_labels()
ax.legend(h1 + h2, l1 + l2, loc="upper left")
plt.tight_layout(); plt.show()

peak_month = int(bln["debit_relatif"].idxmax())
print(f"JAWABAN 4: Bulan dengan debit relatif tertinggi: {peak_month} ({bln['debit_relatif'].max():.2f}x rata-rata); bulan terhujan: {int(bln['hujan'].idxmax())}.")
print(f"- Korelasi Spearman debit dengan hujan harian: {r1:.2f}; dengan akumulasi hujan 7 hari: {r7:.2f} "
      f"({'kuat' if abs(r7) >= 0.5 else 'sedang' if abs(r7) >= 0.3 else 'lemah'}).")

## Tahap 5: Feature Engineering dan Split Data

### 5.1 Feature engineering
**Apa yang terjadi:** dibuat fitur dari debit dan hujan historis, fitur musiman, dan one-hot lokasi. Target adalah debit hari berikutnya dalam skala log.

**Keterangan:**
- Semua fitur dihitung **per kota** dan hanya memakai informasi hari ini dan sebelumnya (tanpa kebocoran masa depan).
- Debit diubah dengan `log1p` agar sungai besar dan kecil berskala sebanding; prediksi dikembalikan ke m³/s dengan `expm1`.
- Kelompok fitur: debit historis (`log_q`, lag 1/2/3/7, selisih, rolling 7 hari), hujan (`rain_mm`, lag, akumulasi 3 dan 7 hari), musiman (`bulan_sin/cos`), lokasi (`loc_*`).

In [ ]:
df = df.sort_values(["location_id", "obs_date"]).reset_index(drop=True)
df["log_q"] = np.log1p(df["discharge_m3s"])
g = df.groupby("location_id")

for k in (1, 2, 3, 7):
    df[f"lq_lag{k}"] = g["log_q"].shift(k)
df["lq_diff1"] = df["log_q"] - df["lq_lag1"]
df["lq_roll7_mean"] = g["log_q"].transform(lambda s: s.rolling(7).mean())
df["lq_roll7_std"] = g["log_q"].transform(lambda s: s.rolling(7).std())

df["rain_lag1"] = g["rain_mm"].shift(1)
df["rain_lag2"] = g["rain_mm"].shift(2)
df["rain_sum3"] = g["rain_mm"].transform(lambda s: s.rolling(3).sum())
df["rain_sum7"] = g["rain_mm"].transform(lambda s: s.rolling(7).sum())

df["bulan_sin"] = np.sin(2 * np.pi * df["obs_date"].dt.month / 12)
df["bulan_cos"] = np.cos(2 * np.pi * df["obs_date"].dt.month / 12)

df["target_log"] = g["log_q"].shift(-1)          # debit besok (skala log)
df["target_q"] = g["discharge_m3s"].shift(-1)    # debit besok (m3/s)

dummies = pd.get_dummies(df["location_id"], prefix="loc", dtype=float)
df = pd.concat([df, dummies], axis=1)

FEAT_Q = ["log_q", "lq_lag1", "lq_lag2", "lq_lag3", "lq_lag7", "lq_diff1", "lq_roll7_mean", "lq_roll7_std"]
FEAT_RAIN = ["rain_mm", "rain_lag1", "rain_lag2", "rain_sum3", "rain_sum7"]
FEAT_SEASON = ["bulan_sin", "bulan_cos"]
FEAT_LOC = list(dummies.columns)
FEATURES = FEAT_Q + FEAT_RAIN + FEAT_SEASON + FEAT_LOC

data = df.dropna(subset=FEATURES + ["target_log"]).copy()
print(f"Baris siap modeling: {len(data)} | jumlah fitur: {len(FEATURES)}")
display(data[FEAT_Q + FEAT_RAIN + ["target_log"]].head())

### 5.2 Split data berdasarkan waktu
**Apa yang terjadi:** data dibagi menurut tanggal: 70% paling awal untuk train, 15% berikutnya untuk validation, 15% paling akhir untuk test. Ambang siaga P90 per kota dihitung dari data train saja.

**Keterangan:** data deret waktu **tidak boleh diacak** karena model akan "mengintip" masa depan dan skornya menjadi bagus secara palsu. Validation dipakai untuk menala model, test hanya untuk evaluasi akhir.

In [ ]:
data = data.sort_values(["obs_date", "location_id"]).reset_index(drop=True)
dates = np.sort(data["obs_date"].unique())
t1 = dates[int(len(dates) * 0.70)]
t2 = dates[int(len(dates) * 0.85)]

train = data[data["obs_date"] < t1].copy()
val = data[(data["obs_date"] >= t1) & (data["obs_date"] < t2)].copy()
test = data[data["obs_date"] >= t2].copy()

display(pd.DataFrame({
    "baris": [len(train), len(val), len(test)],
    "tanggal_awal": [d["obs_date"].min().date() for d in (train, val, test)],
    "tanggal_akhir": [d["obs_date"].max().date() for d in (train, val, test)],
}, index=["train", "validation", "test"]))

thr = train.groupby("location_id")["discharge_m3s"].quantile(0.90)   # ambang siaga P90 (hanya data latih)
print(f"Ambang siaga P90 dihitung untuk {len(thr)} kota.")

## Tahap 6: Pelatihan Model (Gradient Boosting Regressor)

**Apa yang terjadi:** satu algoritma dipakai, yaitu **Gradient Boosting Regressor**. Hyperparameter ditala dengan `GridSearchCV` + `TimeSeriesSplit` pada data train, lalu model terbaik dilatih ulang pada train + validation.

**Keterangan, alasan memilih GBR:**
- Data tabular berukuran ribuan baris dengan fitur campuran (lag, rolling, one-hot): boosting pohon sangat kompetitif.
- Hubungan hujan ke debit non-linear dan berjenjang; pohon menangkapnya tanpa rekayasa interaksi manual.
- Pohon dibangun bertahap untuk memperbaiki galat pohon sebelumnya, sehingga bias lebih rendah.
- Tanpa scaling, ringan dilatih di Colab, dan mendukung feature importance.
- **Keterbatasan:** model pohon tidak bisa memprediksi di atas nilai tertinggi yang pernah ada di data latih, jadi banjir yang melebihi seluruh riwayat akan diprediksi terlalu rendah.

In [ ]:
X_train, y_train = train[FEATURES], train["target_log"]

param_grid = {
    "n_estimators": [200, 400],
    "learning_rate": [0.05, 0.1],
    "max_depth": [2, 3, 4],
    "min_samples_leaf": [3, 10],
}
gs = GridSearchCV(
    GradientBoostingRegressor(subsample=0.8, random_state=RANDOM_STATE),
    param_grid, cv=TimeSeriesSplit(n_splits=3),
    scoring="neg_mean_absolute_error", n_jobs=-1,
)
gs.fit(X_train, y_train)
print("Hyperparameter terbaik:", gs.best_params_)
print(f"MAE validasi silang (skala log): {-gs.best_score_:.4f}")

# Latih ulang pada train + validation (lebih banyak data terbaru)
trainval = pd.concat([train, val])
final_model = clone(gs.best_estimator_).fit(trainval[FEATURES], trainval["target_log"])
print("Model final dilatih pada", len(trainval), "baris (train + validation).")

## Tahap 7: Evaluasi Model

### 7.1 Metrik regresi pada data test
**Apa yang terjadi:** model diuji pada data test yang belum pernah dipakai, lalu dibandingkan dengan **baseline persistensi** (menebak debit besok sama dengan debit hari ini).

**Keterangan:** MAE dan RMSE dalam m³/detik (makin kecil makin baik), R² makin dekat 1 makin baik. Model hanya berguna jika mengalahkan baseline. Skor pooled dipengaruhi sungai besar, karena itu `MAE_%` (galat relatif terhadap rata-rata debit) juga ditampilkan.

In [ ]:
def metrics(y_true_q, y_pred_q):
    mae = mean_absolute_error(y_true_q, y_pred_q)
    return {"MAE": mae,
            "RMSE": float(np.sqrt(mean_squared_error(y_true_q, y_pred_q))),
            "R2": r2_score(y_true_q, y_pred_q),
            "MAE_%": mae / np.mean(y_true_q) * 100}

test["pred_q"] = np.expm1(final_model.predict(test[FEATURES]))
test["pred_base"] = test["discharge_m3s"]
test["tgl_target"] = test["obs_date"] + pd.Timedelta(days=1)

test_res = pd.DataFrame({
    "Baseline (persistensi)": metrics(test["target_q"], test["pred_base"]),
    "Gradient Boosting (final)": metrics(test["target_q"], test["pred_q"]),
}).T.round(3)
display(test_res)

gain = (1 - test_res.iloc[1]["MAE"] / test_res.iloc[0]["MAE"]) * 100
print(f"Perubahan MAE dibanding baseline: {gain:+.1f}% (positif = model lebih baik)")

### 7.2 Grafik aktual vs prediksi
**Apa yang terjadi:** dua grafik dibuat: sebar aktual vs prediksi untuk seluruh kota (kiri), dan deret waktu kota berisiko tertinggi pada periode test (kanan).

**Keterangan:** titik yang menempel garis putus-putus berarti prediksi akurat. Pada deret waktu, perhatikan apakah garis prediksi mengikuti puncak debit, dan apakah lebih dekat ke aktual dibanding baseline abu-abu.

In [ ]:
top_city = risk.index[0] if risk.index[0] in set(test["city_name"]) else test["city_name"].iloc[0]
gc = test[test["city_name"] == top_city].sort_values("tgl_target")

fig, axes = plt.subplots(1, 2, figsize=(16, 5.5))
axes[0].scatter(test["target_q"], test["pred_q"], s=12, alpha=0.5, color="tab:blue")
lim = [test[["target_q", "pred_q"]].min().min(), test[["target_q", "pred_q"]].max().max()]
axes[0].plot(lim, lim, "k--", lw=1)
axes[0].set_xscale("log"); axes[0].set_yscale("log")
axes[0].set_xlabel("Debit aktual (m³/s)"); axes[0].set_ylabel("Debit prediksi (m³/s)")
axes[0].set_title("Aktual vs prediksi, seluruh kota (data test)")

axes[1].plot(gc["tgl_target"], gc["target_q"], color="black", lw=1.6, label="Aktual")
axes[1].plot(gc["tgl_target"], gc["pred_q"], color="tab:red", lw=1.4, ls="--", label="Prediksi GBR")
axes[1].plot(gc["tgl_target"], gc["pred_base"], color="tab:gray", lw=1, label="Baseline")
axes[1].set_title(f"Deret waktu data test: {top_city}")
axes[1].set_ylabel("Debit (m³/s)"); axes[1].tick_params(axis="x", rotation=30); axes[1].legend()
plt.tight_layout(); plt.show()

### 7.3 Keandalan peringatan banjir (klasifikasi ambang P90)
**Apa yang terjadi:** prediksi dan debit aktual diubah menjadi status "Waspada" bila melebihi ambang P90 kota (dari data train). Precision, recall, dan F1 dihitung untuk model dan baseline.

**Keterangan:** *recall* = berapa persen kejadian di atas ambang yang berhasil terdeteksi (penting agar banjir tidak terlewat). *precision* = berapa persen peringatan yang benar (menghindari alarm palsu). F1 adalah keseimbangan keduanya.

In [ ]:
test["ambang"] = test["location_id"].map(thr)
y_true = (test["target_q"] > test["ambang"]).astype(int)
y_model = (test["pred_q"] > test["ambang"]).astype(int)
y_base = (test["pred_base"] > test["ambang"]).astype(int)

cls = pd.DataFrame({
    "Model GBR": [precision_score(y_true, y_model, zero_division=0), recall_score(y_true, y_model, zero_division=0),
                  f1_score(y_true, y_model, zero_division=0)],
    "Baseline": [precision_score(y_true, y_base, zero_division=0), recall_score(y_true, y_base, zero_division=0),
                 f1_score(y_true, y_base, zero_division=0)],
}, index=["Precision", "Recall", "F1"]).round(3)
display(cls)
print(f"Kejadian di atas ambang pada data test: {int(y_true.sum())} dari {len(y_true)} hari-kota\n")
print(classification_report(y_true, y_model, target_names=["Aman", "Waspada"], zero_division=0))

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, yp, title in zip(axes, [y_model, y_base], ["Model GBR", "Baseline"]):
    ConfusionMatrixDisplay(confusion_matrix(y_true, yp, labels=[0, 1]),
                           display_labels=["Aman", "Waspada"]).plot(ax=ax, cmap="Blues", colorbar=False)
    ax.set_title(f"Peringatan banjir: {title}")
plt.tight_layout(); plt.show()

### 7.4 Grafik feature importance
**Apa yang terjadi:** pentingnya fitur diukur dengan dua cara: importance bawaan GBR dan *permutation importance* pada data test. Hasilnya digambar sebagai grafik batang.

**Keterangan:** permutation importance mengacak nilai satu fitur lalu melihat seberapa besar MAE memburuk, sehingga lebih jujur karena diukur pada data baru. Fitur dikelompokkan (debit historis, hujan, musiman, lokasi) untuk melihat sinyal mana yang paling menentukan prediksi.

In [ ]:
perm = permutation_importance(final_model, test[FEATURES], np.log1p(test["target_q"]),
                              scoring="neg_mean_absolute_error", n_repeats=15,
                              random_state=RANDOM_STATE, n_jobs=1)

def group_of(f):
    if f.startswith("loc_"): return "Identitas lokasi"
    if f.startswith("rain"): return "Hujan"
    if f.startswith("bulan"): return "Musiman (bulan)"
    return "Debit historis"

imp = pd.DataFrame({
    "fitur": FEATURES,
    "bawaan": final_model.feature_importances_,
    "permutasi": perm.importances_mean,
    "permutasi_std": perm.importances_std,
})
imp["kelompok"] = imp["fitur"].map(group_of)
imp = imp.sort_values("permutasi", ascending=False).reset_index(drop=True)
display(imp.head(12).round(4))

fig, axes = plt.subplots(1, 3, figsize=(19, 6))
top = imp.head(12).iloc[::-1]
axes[0].barh(top["fitur"], top["bawaan"], color="tab:blue"); axes[0].set_title("Importance bawaan GBR")
axes[1].barh(top["fitur"], top["permutasi"], xerr=top["permutasi_std"], color="tab:red")
axes[1].set_title("Permutation importance (kenaikan MAE log)")
grp = imp.groupby("kelompok")["permutasi"].sum().sort_values()
axes[2].barh(grp.index, grp.values, color="tab:green"); axes[2].set_title("Importance per kelompok fitur")
plt.tight_layout(); plt.show()

print("JAWABAN 5:")
print(f"- Algoritma: Gradient Boosting Regressor | MAE test {test_res.iloc[1]['MAE']:.2f} m³/s vs baseline {test_res.iloc[0]['MAE']:.2f} ({gain:+.1f}%).")
print(f"- Peringatan banjir: precision {cls.loc['Precision','Model GBR']:.2f}, recall {cls.loc['Recall','Model GBR']:.2f}, F1 {cls.loc['F1','Model GBR']:.2f}.")
print(f"- 3 fitur terpenting: {', '.join(imp['fitur'].head(3))} | kelompok terkuat: {grp.idxmax()}.")

## Tahap 8: Menyimpan Aset Model untuk Dashboard Streamlit

**Apa yang terjadi:** semua keperluan dashboard disimpan ke folder `dashboard_assets` (di folder yang sama dengan data, jadi di Google Drive bila data di Drive), lalu di-zip.

**Keterangan, isi folder:**

| File | Kegunaan di dashboard |
|---|---|
| `model_gbr_debit_sungai.joblib` | Model terlatih untuk prediksi |
| `model_meta.json` | Daftar fitur, ambang P90 per kota, metrik, versi library |
| `data_fitur_harian.csv` | Riwayat harian per kota lengkap dengan fitur (untuk grafik dan prediksi ulang) |
| `forecast_terbaru.csv` | Prediksi debit besok per kota dan status peringatan |
| `hasil_prediksi_test.csv` | Aktual vs prediksi pada data test |
| `ringkasan_kota.csv`, `skor_risiko_kota.csv` | Tabel EDA 1 dan EDA 2 |
| `feature_importance.csv`, `metrik_evaluasi.csv` | Halaman evaluasi model |
| `lokasi_kota.csv` | Koordinat untuk peta |
| `requirements_dashboard.txt` | Versi library yang sama dengan saat model dilatih |

Versi `scikit-learn` pada dashboard **harus sama** dengan versi saat model disimpan, jika tidak file `.joblib` bisa gagal dimuat atau memberi hasil berbeda.

In [ ]:
ASSET_DIR = os.path.join(BASE_DIR, "dashboard_assets")
os.makedirs(ASSET_DIR, exist_ok=True)

# 1. Model
joblib.dump(final_model, os.path.join(ASSET_DIR, "model_gbr_debit_sungai.joblib"))

# 2. Prediksi terbaru per kota (hari data terakhir -> debit besok)
latest = df[df["target_log"].isna()].dropna(subset=FEATURES).copy()
latest["prediksi_debit_besok"] = np.expm1(final_model.predict(latest[FEATURES]))
latest["ambang_siaga_P90"] = latest["location_id"].map(thr)
latest["status"] = np.where(latest["prediksi_debit_besok"] > latest["ambang_siaga_P90"], "WASPADA", "Normal")
forecast = latest[["location_id", "city_name", "river_name", "province", "latitude", "longitude",
                   "obs_date", "discharge_m3s", "prediksi_debit_besok", "ambang_siaga_P90", "status"]].rename(
    columns={"obs_date": "tanggal_data", "discharge_m3s": "debit_hari_ini"})
forecast.to_csv(os.path.join(ASSET_DIR, "forecast_terbaru.csv"), index=False)

# 3. Data harian + fitur, hasil test, tabel EDA, importance, metrik, lokasi
ID_COLS = ["location_id", "city_name", "river_name", "province", "latitude", "longitude",
           "obs_date", "discharge_m3s", "rain_mm"]
df[ID_COLS + FEATURES + ["target_q"]].to_csv(os.path.join(ASSET_DIR, "data_fitur_harian.csv"), index=False)
test[["location_id", "city_name", "obs_date", "tgl_target", "target_q", "pred_q", "pred_base", "ambang"]].to_csv(
    os.path.join(ASSET_DIR, "hasil_prediksi_test.csv"), index=False)
summary.reset_index().to_csv(os.path.join(ASSET_DIR, "ringkasan_kota.csv"), index=False)
risk.reset_index().to_csv(os.path.join(ASSET_DIR, "skor_risiko_kota.csv"), index=False)
imp.to_csv(os.path.join(ASSET_DIR, "feature_importance.csv"), index=False)
pd.concat([test_res.assign(jenis="regresi"), cls.T.assign(jenis="peringatan")]).reset_index().rename(
    columns={"index": "model"}).to_csv(os.path.join(ASSET_DIR, "metrik_evaluasi.csv"), index=False)
df_loc[df_loc["location_id"].isin(df["location_id"].unique())].to_csv(os.path.join(ASSET_DIR, "lokasi_kota.csv"), index=False)

# 4. Metadata
params = final_model.get_params()
meta = {
    "dibuat": datetime.now().isoformat(timespec="seconds"),
    "model": "GradientBoostingRegressor",
    "hyperparameter": {k: params[k] for k in ("n_estimators", "learning_rate", "max_depth", "min_samples_leaf", "subsample")},
    "target": "log1p(debit hari berikutnya); debit (m3/s) = expm1(prediksi)",
    "fitur": FEATURES,
    "kelompok_fitur": {"debit_historis": FEAT_Q, "hujan": FEAT_RAIN, "musiman": FEAT_SEASON, "lokasi": FEAT_LOC},
    "ambang_siaga_p90_m3s": {k: float(v) for k, v in thr.items()},
    "kota": df[["location_id", "city_name"]].drop_duplicates().set_index("location_id")["city_name"].to_dict(),
    "periode_data": [str(df["obs_date"].min().date()), str(df["obs_date"].max().date())],
    "periode_latih": [str(train["obs_date"].min().date()), str(val["obs_date"].max().date())],
    "periode_uji": [str(test["obs_date"].min().date()), str(test["obs_date"].max().date())],
    "metrik_test": {k: float(v) for k, v in test_res.iloc[1].items()},
    "metrik_baseline_test": {k: float(v) for k, v in test_res.iloc[0].items()},
    "metrik_peringatan": {k: float(v) for k, v in cls["Model GBR"].items()},
    "ringkasan_eda": {"kota_paling_berisiko": str(risk.index[0]),
                      "rata_hari_menuju_puncak": float(peak_day.mean()),
                      "bulan_debit_tertinggi": int(peak_month),
                      "korelasi_hujan7h_debit": float(r7)},
    "versi": {"scikit-learn": sklearn.__version__, "pandas": pd.__version__, "numpy": np.__version__,
              "joblib": joblib.__version__},
}
with open(os.path.join(ASSET_DIR, "model_meta.json"), "w") as f:
    json.dump(meta, f, indent=2, ensure_ascii=False)

# 5. Requirements dashboard (versi disamakan dengan lingkungan pelatihan)
reqs = ["streamlit", "plotly", f"pandas=={pd.__version__}", f"numpy=={np.__version__}",
        f"scikit-learn=={sklearn.__version__}", f"joblib=={joblib.__version__}"]
with open(os.path.join(ASSET_DIR, "requirements_dashboard.txt"), "w") as f:
    f.write("\n".join(reqs) + "\n")

# 6. Zip dan unduh
zip_path = shutil.make_archive(os.path.join(BASE_DIR, "dashboard_assets"), "zip", ASSET_DIR)
print("Aset tersimpan di:", ASSET_DIR)
for fn in sorted(os.listdir(ASSET_DIR)):
    print(f" - {fn:34s} {os.path.getsize(os.path.join(ASSET_DIR, fn)) / 1024:8.1f} KB")
print("ZIP:", zip_path)
if IN_COLAB:
    files.download(zip_path)

### Uji muat ulang aset
**Apa yang terjadi:** model dan metadata dibaca kembali dari disk lalu dipakai memprediksi, persis seperti yang nanti dilakukan aplikasi Streamlit.

**Keterangan:** hasilnya harus sama dengan tabel `forecast_terbaru.csv`. Jika ya, aset siap dipakai dashboard.

In [ ]:
model_dash = joblib.load(os.path.join(ASSET_DIR, "model_gbr_debit_sungai.joblib"))
meta_dash = json.load(open(os.path.join(ASSET_DIR, "model_meta.json")))
data_dash = pd.read_csv(os.path.join(ASSET_DIR, "data_fitur_harian.csv"), parse_dates=["obs_date"])

# Ambil hari terakhir tiap kota, hitung prediksi dengan model yang dimuat ulang
last = data_dash.sort_values("obs_date").groupby("location_id").tail(1).dropna(subset=meta_dash["fitur"])
pred = np.expm1(model_dash.predict(last[meta_dash["fitur"]]))

cek = pd.read_csv(os.path.join(ASSET_DIR, "forecast_terbaru.csv")).set_index("location_id")
selisih = np.abs(pred - cek.loc[last["location_id"], "prediksi_debit_besok"].values).max()
print(f"Selisih maksimum prediksi (dimuat ulang vs tersimpan): {selisih:.6f} m³/s")
display(cek[["city_name", "tanggal_data", "debit_hari_ini", "prediksi_debit_besok", "ambang_siaga_P90", "status"]]
        .sort_values("status", ascending=False).head(10).round(2))
print("Aset dashboard siap dipakai.")